# Week 10 live workspace: LLM evaluation

This is the notebook we will code in on Monday. It contains the imports, a synthetic fixture, and safe configuration; we will write the schema, validator, baseline, comparison, and interpretation together.

**Live API calls are off by default.** The offline fixture runs without credentials and contains no real student or campus-service data.


## Learning goals

By the end of class, you should be able to describe and implement this chain:

`defined task → structured response → semantic validation → same-data comparison → inspected errors → claim that fits the evidence`


In [ ]:
import json
import os

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

ALLOWED_LABELS = (
    "transit",
    "study_space",
    "accessibility",
    "food",
    "safety",
    "services",
)

LIVE_MODE = os.getenv("COMPSS211_LIVE_API", "0") == "1"
APPROVED_MODEL_ID = os.getenv("COMPSS211_GEMINI_MODEL", "").strip()


## Supplied synthetic fixture

We wrote these records for class. They cannot support claims about Berkeley students or campus operations. The recorded predictions are sanitized offline examples, not fresh API results.


In [ ]:
FIXTURE = json.loads(r'''{
  "training": [
    {
      "document_id": "TR01",
      "text": "The campus shuttle skipped two stops this morning.",
      "human_label": "transit"
    },
    {
      "document_id": "TR02",
      "text": "Bus arrivals need to be more frequent after class.",
      "human_label": "transit"
    },
    {
      "document_id": "TR03",
      "text": "The night shuttle route no longer reaches my stop.",
      "human_label": "transit"
    },
    {
      "document_id": "TR04",
      "text": "Transit service is delayed near the residence halls.",
      "human_label": "transit"
    },
    {
      "document_id": "TR05",
      "text": "Please add another bus at the evening rush.",
      "human_label": "transit"
    },
    {
      "document_id": "SS01",
      "text": "The library reading room has no open desks.",
      "human_label": "study_space"
    },
    {
      "document_id": "SS02",
      "text": "We need more quiet study rooms during finals.",
      "human_label": "study_space"
    },
    {
      "document_id": "SS03",
      "text": "Group study tables are always occupied.",
      "human_label": "study_space"
    },
    {
      "document_id": "SS04",
      "text": "The study lounge closes too early.",
      "human_label": "study_space"
    },
    {
      "document_id": "SS05",
      "text": "Library seating is difficult to find at noon.",
      "human_label": "study_space"
    },
    {
      "document_id": "AC01",
      "text": "The elevator is broken and the stairs are the only route.",
      "human_label": "accessibility"
    },
    {
      "document_id": "AC02",
      "text": "The wheelchair ramp is blocked by construction fencing.",
      "human_label": "accessibility"
    },
    {
      "document_id": "AC03",
      "text": "The automatic door button does not work.",
      "human_label": "accessibility"
    },
    {
      "document_id": "AC04",
      "text": "Captions were missing from the event video.",
      "human_label": "accessibility"
    },
    {
      "document_id": "AC05",
      "text": "The accessible entrance is difficult to locate.",
      "human_label": "accessibility"
    },
    {
      "document_id": "FO01",
      "text": "The dining hall ran out of vegetarian meals.",
      "human_label": "food"
    },
    {
      "document_id": "FO02",
      "text": "Campus food options close before evening classes end.",
      "human_label": "food"
    },
    {
      "document_id": "FO03",
      "text": "The cafe needs clearer allergen labels.",
      "human_label": "food"
    },
    {
      "document_id": "FO04",
      "text": "Lunch lines at the dining hall are too long.",
      "human_label": "food"
    },
    {
      "document_id": "FO05",
      "text": "There are few affordable meals on campus.",
      "human_label": "food"
    },
    {
      "document_id": "SA01",
      "text": "The walkway is dark and feels unsafe at night.",
      "human_label": "safety"
    },
    {
      "document_id": "SA02",
      "text": "A broken light leaves the path unlit after dark.",
      "human_label": "safety"
    },
    {
      "document_id": "SA03",
      "text": "The emergency phone near the garage is not working.",
      "human_label": "safety"
    },
    {
      "document_id": "SA04",
      "text": "More lighting would make the late walk safer.",
      "human_label": "safety"
    },
    {
      "document_id": "SA05",
      "text": "The crosswalk signal is dangerous for pedestrians.",
      "human_label": "safety"
    },
    {
      "document_id": "SV01",
      "text": "The advising portal will not let me book an appointment.",
      "human_label": "services"
    },
    {
      "document_id": "SV02",
      "text": "My student ID replacement request is still pending.",
      "human_label": "services"
    },
    {
      "document_id": "SV03",
      "text": "The financial aid office has not answered my message.",
      "human_label": "services"
    },
    {
      "document_id": "SV04",
      "text": "The registration website keeps rejecting the form.",
      "human_label": "services"
    },
    {
      "document_id": "SV05",
      "text": "I cannot reach anyone at the campus help desk.",
      "human_label": "services"
    }
  ],
  "evaluation": [
    {
      "document_id": "EV01",
      "text": "The shuttle stopped arriving after 9 p.m.",
      "human_label": "transit"
    },
    {
      "document_id": "EV02",
      "text": "Love waiting forty minutes for the bus 🙃",
      "human_label": "transit"
    },
    {
      "document_id": "EV03",
      "text": "Every seat in the library is taken again.",
      "human_label": "study_space"
    },
    {
      "document_id": "EV04",
      "text": "So quiet in the reading room—unless you count the drilling.",
      "human_label": "study_space"
    },
    {
      "document_id": "EV05",
      "text": "The elevator outage blocks the only step-free route.",
      "human_label": "accessibility"
    },
    {
      "document_id": "EV06",
      "text": "The access office form is not readable by my screen reader.",
      "human_label": "accessibility"
    },
    {
      "document_id": "EV07",
      "text": "Please label allergens on the cafe menu.",
      "human_label": "food"
    },
    {
      "document_id": "EV08",
      "text": "Dinner service ends before my evening seminar.",
      "human_label": "food"
    },
    {
      "document_id": "EV09",
      "text": "It is not safe after 9 on the unlit path.",
      "human_label": "safety"
    },
    {
      "document_id": "EV10",
      "text": "This crosswalk is SAFE?! Cars never stop.",
      "human_label": "safety"
    },
    {
      "document_id": "EV11",
      "text": "The advising appointment page keeps crashing.",
      "human_label": "services"
    },
    {
      "document_id": "EV12",
      "text": "Where is my replacement campus ID card?",
      "human_label": "services"
    }
  ],
  "recorded_predictions": [
    {
      "document_id": "EV01",
      "label": "transit",
      "rationale": "The comment concerns the operating hours of the shuttle."
    },
    {
      "document_id": "EV02",
      "label": "services",
      "rationale": "The writer complains about a long wait for a campus service."
    },
    {
      "document_id": "EV03",
      "label": "study_space",
      "rationale": "The comment concerns the availability of library seating."
    },
    {
      "document_id": "EV04",
      "label": "safety",
      "rationale": "The drilling is described as a disruptive condition in the room."
    },
    {
      "document_id": "EV05",
      "label": "accessibility",
      "rationale": "A broken elevator prevents use of a step-free route."
    },
    {
      "document_id": "EV06",
      "label": "services",
      "rationale": "The comment concerns an online form operated by a campus office."
    },
    {
      "document_id": "EV07",
      "label": "food",
      "rationale": "The request is about allergen information on a cafe menu."
    },
    {
      "document_id": "EV08",
      "label": "food",
      "rationale": "The comment concerns the hours of dinner service."
    },
    {
      "document_id": "EV09",
      "label": "safety",
      "rationale": "The comment explicitly describes an unsafe unlit path at night."
    },
    {
      "document_id": "EV10",
      "label": "safety",
      "rationale": "The sarcastic wording describes cars failing to stop at a crosswalk."
    },
    {
      "document_id": "EV11",
      "label": "services",
      "rationale": "The advising appointment system is failing."
    },
    {
      "document_id": "EV12",
      "label": "services",
      "rationale": "The comment asks about a campus ID replacement service."
    }
  ]
}''')

training = pd.DataFrame(FIXTURE["training"])
evaluation = pd.DataFrame(FIXTURE["evaluation"])
recorded_predictions = pd.DataFrame(FIXTURE["recorded_predictions"])

assert training["document_id"].is_unique
assert evaluation["document_id"].is_unique
assert recorded_predictions["document_id"].is_unique
assert set(training["human_label"]).issubset(ALLOWED_LABELS)
assert set(evaluation["human_label"]).issubset(ALLOWED_LABELS)
assert set(evaluation["document_id"]) == set(recorded_predictions["document_id"])


## Live coding area

We will build each stage from the contract rather than accepting model output directly.


In [ ]:
# 1. Define the response schema and semantic validator here.


In [ ]:
# 2. Fit the deterministic TF-IDF baseline here.


In [ ]:
# 3. Validate and join the recorded predictions here.


In [ ]:
# 4. Compare both methods on identical documents and inspect errors here.


In [ ]:
# 5. Record provenance, token use, cost, and a stopping decision here.


## Before Friday

Review the companion reference guide. Bring one sentence explaining why valid JSON is necessary but insufficient for valid research data.
